<a href="https://colab.research.google.com/github/RohanYashraj/ifoa-workshop/blob/main/notebooks_v2/01_genai_basics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 01 · GenAI Basics — your first calls to the reasoner

**Agentic AI for Actuaries** · IFoA Workshop · 10 July 2026 · Hub: `github.com/rohanyashraj/ifoa-workshop`

> All data in this notebook is **hypothetical** — ABC Insurer is a fictional entity calibrated to plausible Indian market experience, for teaching only.

**Used in:** Session 1, Part 2 (The Reasoner).
**You will:** make your first Gemini API call, practise the CCCE prompt discipline, watch a hallucination happen on demand, and get guaranteed-parseable JSON out of an LLM.

**Setup (2 minutes):**
1. Get a free Gemini API key at [aistudio.google.com](https://aistudio.google.com) → *Get API key*.
2. In Colab, click the **key icon** (left sidebar) → *Add new secret* → name it `GOOGLE_API_KEY`, paste the key, toggle notebook access ON.
3. Run the cells top to bottom (`Runtime → Run all` after setup).

In [3]:
%pip install -q -U google-genai google-auth==2.49.0

## §1 · Auth — the key never appears in the notebook
Colab Secrets keeps the key out of the notebook file. This is the same hygiene you will use for every agent you ship: secrets live in a store, never in code.

In [4]:
import os
from google import genai
from IPython.display import Markdown, display
from google.colab import userdata   # Colab-only; see comment below for local Jupyter

os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
# Local Jupyter alternative:
#   os.environ["GOOGLE_API_KEY"] = "..."  # or use python-dotenv

client = genai.Client()
MODEL = "gemini-3.1-flash-lite"   # PINNED — silent model drift is an audit failure
print("Client ready, model pinned to:", MODEL)

Client ready, model pinned to: gemini-3.1-flash-lite


## §2 · First call — define IBNR for a board member

In [5]:
response = client.models.generate_content(
    model=MODEL,
    contents="Define IBNR for a non-actuarial board member, in one line.",
)
# Clear visual separation
print("=" * 70)
print("📋 GEMINI MODEL RESPONSE")
print("=" * 70)

display(Markdown(response.text))

print("\n" + "=" * 70)
print("END OF MODEL RESPONSE")
print("=" * 70)
print("\n--- usage ---")
print(response.usage_metadata)   # token counts: you will care about these when agents multiply call volume

📋 GEMINI MODEL RESPONSE


IBNR (Incurred But Not Reported) represents the estimated financial reserve a company must set aside to cover claims that have already occurred but have not yet been filed by the policyholder.


END OF MODEL RESPONSE

--- usage ---
cache_tokens_details=None cached_content_token_count=None candidates_token_count=38 candidates_tokens_details=None prompt_token_count=19 prompt_tokens_details=[ModalityTokenCount(
  modality=<MediaModality.TEXT: 'TEXT'>,
  token_count=19
)] thoughts_token_count=None tool_use_prompt_token_count=None tool_use_prompt_tokens_details=None total_token_count=57 traffic_type=None


## §3 · CCCE — Clarity, Context, Constraints, Examples
The prompt below is the worked example from the slides: an IBNR commentary for ABC Health Q3 2024. Each bracketed fragment does exactly one job — edit any part without breaking the others.

**Exercise:** delete the Constraints block, re-run, and compare. Then rewrite the prompt for *your* line of business.

In [6]:
ccce_prompt = """
[Clarity] Write a two-paragraph commentary on the IBNR result for ABC Health Q3 2024.
[Context] Indemnity health book. Chain-ladder ultimate INR 186 Cr vs prior estimate INR 172 Cr.
Q3 saw a hospital network strike in two states.
[Constraints] Audience: appointed actuary peer-review meeting. Max 180 words.
Do not invent figures. Cite only the figures provided above.
[Example] Voice to match: "The Q2 ultimate of INR 164 Cr increased to INR 172 Cr after the network
expansion in Tier 2 cities..."
"""
resp = client.models.generate_content(model=MODEL, contents=ccce_prompt)
# Clear visual separation
print("=" * 70)
print("📋 GEMINI MODEL RESPONSE")
print("=" * 70)

display(Markdown(resp.text))

print("\n" + "=" * 70)
print("END OF MODEL RESPONSE")
print("=" * 70)

📋 GEMINI MODEL RESPONSE


The Q3 2024 IBNR valuation for the indemnity health book reflects an upward revision in ultimate claims, moving from the prior estimate of INR 172 Cr to INR 186 Cr. This development is primarily driven by emerging experience following the hospital network strike across two states during the quarter. The disruption led to delayed reporting patterns and shifting utilization behaviors, necessitating a strengthening of the reserves to ensure adequacy against these emerging claim trends.

The increase of INR 14 Cr underscores the sensitivity of the chain-ladder projection to the recent operational volatility within the network. While the model remains the primary basis for the ultimate estimate, the strike-related distortions warrant a cautious interpretation of the latest development factors. We recommend maintaining these elevated reserve levels while monitoring the subsequent settlement activity to confirm if the reporting lag normalizes as the network stabilizes post-disruption.


END OF MODEL RESPONSE


### §3.1 · Demo 1 — the vague version, for contrast
Run the deliberately vague prompt below, then re-run the CCCE version above and **diff the outputs**. Same model, same cost — the entire quality delta is the prompt.

In [7]:
vague = "Write about IBNR for our board."
# Clear visual separation
print("=" * 70)
print("📋 GEMINI MODEL RESPONSE")
print("=" * 70)

display(Markdown(client.models.generate_content(model=MODEL, contents=vague).text[:800]))

print("\n" + "=" * 70)
print("END OF MODEL RESPONSE")
print("=" * 70)
# Expect: a generic essay that INVENTS plausible numbers (we gave it none)
# and lands in a register somewhere between textbook and LinkedIn.

📋 GEMINI MODEL RESPONSE


This briefing note is designed for a Board of Directors. It balances technical accuracy with the strategic oversight required at the governance level.

***

# Board Briefing: Understanding IBNR (Incurred But Not Reported)

## 1. Executive Summary
IBNR stands for **Incurred But Not Reported**. In the context of insurance and risk management, it represents an estimate of the liability for claims that have already occurred but have not yet been reported to the company. Because these claims are "hidden" in the timeline between the incident and the notification, IBNR is a critical component of our loss reserves and a significant variable in our balance sheet health.

## 2. The Mechanics: Why does IBNR exist?
There is almost always a "time lag" between the occurrence of an event and the filing o


END OF MODEL RESPONSE


### §3.2 · Demo 2 — one fact, two audiences
Audience is a prompt parameter. Same reserve-strengthening fact, rendered for a board member and for a new student. Note: the model *dresses* the fact we supply — it does not source it.

In [8]:
fact = ("We strengthened motor BI reserves by INR 42 Cr "
        "following the new tribunal award benchmarks.")

for audience, style in [
    ("board member", "2 sentences, business impact first, no jargon"),
    ("new actuarial student", "4 sentences, explain WHY tribunal awards drive BI reserves, define terms"),
]:
    r = client.models.generate_content(
        model=MODEL,
        contents=f"Explain: {fact} For a {audience}. {style}")
    # Clear visual separation
    print("=" * 70)
    print("📋 GEMINI MODEL RESPONSE")
    print("=" * 70)

    display(Markdown(f"**{audience.upper()}**\n\n{r.text}"))

    print("\n" + "=" * 70)
    print("END OF MODEL RESPONSE")
    print("=" * 70)

📋 GEMINI MODEL RESPONSE


**BOARD MEMBER**

We have increased our financial reserves by INR 42 crore to ensure we can fully cover higher settlement costs mandated by recent court rulings. This proactive adjustment protects our balance sheet from future volatility and ensures we remain prepared for the evolving legal landscape in motor insurance.


END OF MODEL RESPONSE
📋 GEMINI MODEL RESPONSE


**NEW ACTUARIAL STUDENT**

Bodily Injury (BI) reserves represent the estimated funds an insurer must set aside to cover future payouts for physical harm claims, such as medical costs or loss of earnings. Tribunal awards act as the "settlement benchmark" because they establish the legal standard for how much compensation is owed to a victim for specific types of injuries. When a tribunal increases these award benchmarks, it signals that previous cost assumptions are now too low to cover the legal liability for similar outstanding claims. Consequently, the insurer must "strengthen" reserves—increasing the held capital—to ensure the company remains solvent and prepared to pay these higher-than-anticipated claim values.


END OF MODEL RESPONSE


### §3.3 · Demo 3 — few-shot examples tame formatting
Show, don't tell: two worked examples buy you the delimiter, the casing, the arrow convention, and no chatty preamble. **Exercise:** feed it a genuinely weird input and see whether the pattern holds.

In [9]:
prompt = """Convert each change to the format of the examples.

EXAMPLES
In: We moved lapse from 6% to 5.5% for durations 2+.
Out: LAPSE | dur 2+ | 6.0% -> 5.5%
In: Expense inflation up 50bps.
Out: EXPENSE_INFL | all | +50bps

NOW CONVERT
In: Mortality improvement for males 45-60 moves from 1.5% to 1.25%.
Out:"""
print(client.models.generate_content(model=MODEL, contents=prompt).text)

MORTALITY | male 45-60 | 1.5% -> 1.25%


### §3.4 · Demo 4 — step-by-step reasoning (with a warning label)
Asking for steps improves reliability — it does **not** guarantee it. Re-run this cell three times: do the running totals stay identical? This is why the afternoon's agent does arithmetic in *Python* and lets Gemini narrate.

In [10]:
prompt = """A motor policy has base premium INR 6,500 with relativities:
vehicle age 6-9yrs = 1.15, SUV = 1.20, Tier2 = 1.00, NCB 35% = 0.65.
Walk through the premium calculation STEP BY STEP, showing the running
total after each factor, then state the final premium."""
# Clear visual separation
print("=" * 70)
print("📋 GEMINI MODEL RESPONSE")
print("=" * 70)

display(Markdown(client.models.generate_content(model=MODEL, contents=prompt).text))

print("\n" + "=" * 70)
print("END OF MODEL RESPONSE")
print("=" * 70)
# Check by hand: 6500 * 1.15 * 1.20 * 1.00 * 0.65 = 5,830.50

📋 GEMINI MODEL RESPONSE


To calculate the final premium, we apply each relativity factor sequentially to the base premium. 

**Base Premium: INR 6,500**

### Step-by-Step Calculation:

**Step 1: Apply Vehicle Age (6-9 yrs)**
*   Calculation: $6,500 \times 1.15$
*   **Running Total: INR 7,475**

**Step 2: Apply Vehicle Type (SUV)**
*   Calculation: $7,475 \times 1.20$
*   **Running Total: INR 8,970**

**Step 3: Apply Geographic Location (Tier 2)**
*   Calculation: $8,970 \times 1.00$
*   **Running Total: INR 8,970**

**Step 4: Apply No Claim Bonus (NCB 35%)**
*   Calculation: $8,970 \times 0.65$
*   **Running Total: INR 5,830.50**

***

### Final Premium:
The final calculated premium for the motor policy is **INR 5,830.50**.


END OF MODEL RESPONSE


### §3.5 · Demo review — the habit that IS the skill
1. CCCE moved quality more than a model upgrade would — specification beats horsepower.
2. Register control is leverage, but the model dresses facts; it doesn't source them.
3. Few-shot is a formatting contract — stress-test it before relying on it.
4. Step-by-step is transparency, not verified arithmetic.

**The loop:** prompt → output → review → edit prompt — the same loop you'll run on agent traces this afternoon.

## §4 · The hallucination demo — run it, believe it
We ask for a regulation that **does not exist**. The model will not say 'no such factor' — it will produce the most *plausible-sounding* answer, confidently.

⚠️ This exact failure mode reappears **inside your agent** in notebook 04 — and you will fix it with a guardrail tool.

In [11]:
hallucination_prompt = (
    "What is the IRDAI motor tariff factor for hatchbacks under 1000cc? "
    "Give the exact factor value and the section reference."
)
resp = client.models.generate_content(model=MODEL, contents=hallucination_prompt)
# Clear visual separation
print("=" * 70)
print("📋 GEMINI MODEL RESPONSE")
print("=" * 70)

display(Markdown(resp.text))

print("\n" + "=" * 70)
print("END OF MODEL RESPONSE")
print("=" * 70)
print("\n⚠️  Verify before you trust: there is no such published factor. "
      "Whatever appears above was constructed to be plausible, not true.")

📋 GEMINI MODEL RESPONSE


To provide an accurate answer, it is important to clarify a regulatory distinction: **The IRDAI does not currently set a "tariff factor"** for motor insurance pricing.

Since the de-tariffing of the motor insurance sector in India on **January 1, 2007**, insurance companies are free to determine their own premium rates (the "Own Damage" component) based on their internal actuarial data. Therefore, there is no standardized industry-wide factor for hatchbacks under 1000cc for the Own Damage section.

However, the **Third-Party (TP) Liability** premium is still regulated by the IRDAI. 

### 1. The Statutory Rate (Third-Party Liability)
For private cars (including hatchbacks) with an engine capacity **not exceeding 1000cc**, the IRDAI sets a fixed premium rate. 

*   **Current Premium Rate:** **₹2,094** (as per the latest notification).
*   **Section Reference:** This is governed by the **IRDAI (Regulatory Sandbox) Regulations and subsequent annual gazette notifications** issued by the Ministry of Road Transport and Highways (MoRTH) in consultation with the IRDAI. The most recent notification is the **"Notification S.O. 2423(E) dated 25th May 2022,"** which established the current slab-based TP rates.

### 2. Regarding "Factors" (IDV and Depreciation)
If you are looking for the "factor" used to calculate the value of the car or depreciation, those are defined in the **India Motor Tariff (IMT)**:

*   **IDV (Insured Declared Value):** The IDV is calculated based on the manufacturer’s listed selling price, adjusted for depreciation based on the age of the vehicle. 
*   **Depreciation Schedule (Section 3 of the IMT):**
    *   Not exceeding 6 months: 5%
    *   Exceeding 6 months but not exceeding 1 year: 15%
    *   Exceeding 1 year but not exceeding 2 years: 20%
    *   Exceeding 2 years but not exceeding 3 years: 30%
    *   Exceeding 3 years but not exceeding 4 years: 40%
    *   Exceeding 4 years but not exceeding 5 years: 50%

### Summary
*   **Own Damage Factor:** No fixed IRDAI factor; determined by the insurer's underwriting guidelines (based on geography, claims experience, and vehicle make/model).
*   **Third-Party Premium:** Fixed at **₹2,094** for vehicles ≤ 1000cc.
*   **Reference:** The **India Motor Tariff (IMT)** for general rules and **MoRTH Notification S.O. 2423(E)** for current TP pricing.

***Disclaimer:** Insurance rates are subject to change by government notification. Please check the latest IRDAI circulars or the [official MoRTH website](https://morth.nic.in/) for real-time updates.*


END OF MODEL RESPONSE

⚠️  Verify before you trust: there is no such published factor. Whatever appears above was constructed to be plausible, not true.


## §5 · Structured output — because agents speak JSON
One config line guarantees parseable JSON. This is how every component of an agentic system exchanges data — prose is only for humans at the last step.

In [12]:
import json

prompt = """For private car comprehensive insurance, list 5 rating factors.
For each: name, direction (increase/decrease premium), one-line justification. Return JSON."""

resp = client.models.generate_content(
    model=MODEL,
    contents=prompt,
    config={"response_mime_type": "application/json"},
)
factors = json.loads(resp.text)   # guaranteed to parse
for f in factors:
    print(f)

{'name': 'Driver Age', 'direction': 'decrease', 'justification': 'Younger, inexperienced drivers are statistically more likely to be involved in accidents than older, more experienced drivers.'}
{'name': 'Vehicle Value', 'direction': 'increase', 'justification': 'Higher-valued vehicles result in higher repair or replacement costs for the insurer in the event of a total loss or collision.'}
{'name': 'No Claims Bonus', 'direction': 'decrease', 'justification': 'A history of driving without making claims indicates a lower risk profile and rewards policyholders for safe driving behavior.'}
{'name': 'Geographic Location', 'direction': 'increase', 'justification': 'Areas with high traffic density, crime rates, or historical accident frequencies carry a higher risk of claims.'}
{'name': 'Vehicle Performance', 'direction': 'increase', 'justification': 'High-performance vehicles are often associated with higher speed capabilities and more expensive specialized repair parts.'}


## §6 · Review exercise — mark the model's homework
Treat the JSON above as a junior analyst's first draft and grade it:

1. Is every **direction** consistent with your priors?
2. Did it name factors your book doesn't collect (e.g. telematics, annual mileage)?
3. What material factors are **missing** (vehicle make? segment?)
4. What would you still need before any of this goes near a tariff filing? *(Hint: magnitudes → a GLM run → notebook 02.)*

**The rule that survives today:** the reasoner narrates; tools know; humans sign.

---
**Log what you ran.** For anything regulatory: save the full prompt–response pair, the model id, and the timestamp — 'the AI wrote it' is not a defence without the receipt.

In [13]:
# Minimal call log — one CSV row per call. In production this is your observability stack.
import datetime, csv, pathlib

def log_call(prompt, response_text, model=MODEL, path="genai_call_log.csv"):
    new = not pathlib.Path(path).exists()
    with open(path, "a", newline="") as f:
        w = csv.writer(f)
        if new:
            w.writerow(["ts_utc", "model", "prompt", "response"])
        w.writerow([datetime.datetime.utcnow().isoformat(), model, prompt, response_text])

log_call(prompt, resp.text)
print("logged — this habit is checklist question 10 in miniature")

logged — this habit is checklist question 10 in miniature


/tmp/ipykernel_2968/3299675427.py:10: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  w.writerow([datetime.datetime.utcnow().isoformat(), model, prompt, response_text])
